# Assignment 5

Deadline: xx.xx.2026 12:00 CEST

## Task

Develop an investment strategy for the Swiss equity market, backtest it using the provided datasets (i.e., `market_data.parquet`, `jkp_data.parquet`, `spi_index.csv`) and analyze its performance by benchmarking it against the Swiss Performance Index (SPI). Build on the existing qpmwp-course codebase and extend it with any additional components required. Summarize your approach and findings in a written report.

### Coding (15 points)



- Selection:
  Use some of the existing selection item builder functions (in `bibfn_selection.py`, applied via `SelectionItemBuilder`) to filter stocks based on specific criteria, and/or implement your own filters (e.g., exclude low-quality or high-volatility stocks).

- Optimization Data & Constraints:
  Use the implemented functions for preparing optimization inputs (in `bibfn_optimization_data.py`, applied via `OptimizationItemBuilder`) and for specifying constraints such as stock, sector, or factor exposure limits (in `bibfn_constraints.py`, applied via `OptimizationItemBuilder`). Extend these with your own functions where appropriate.

- Signal Generation:
  Use a machine learning method to estimate optimization inputs such as expected returns or risk. Possible approaches include regression, classification, or learning-to-rank models. A good starting point is the demo notebook `xsection_regressor.ipynb` which shows how to generate predictive signals. Use jkp_data as features or engineer your own (e.g., technical indicators from returns or prices). Alternatively, or in combination, you may use a factor model to construct the optimization inputs.

- Optimization Model:
  Use an optimization model to determine portfolio weights. You may rely on an existing class (e.g., `MeanVariance`, `LeastSquares`, or `BlackLitterman`) or implement a custom model. If you choose to create a custom optimization model, develop a class inheriting from `Optimization` and ensure implement the methods `set_objective` (to construct the coefficients of the objective function) and `solve` (to run the optimization).

- Simulation:
  Backtest the strategy and simulate portfolio returns, incorporating fixed costs of 1% per year and transaction costs of 0.2% per rebalancing.


### Report (15 points):

Produce an HTML report (for example, by converting a .ipynb notebook to HTML) containing:

- High-level strategy overview: A clear description of the investment strategy and its rationale.

- Detailed explanation of the backtesting steps: A step-by-step explanation of the backtest design, including model choices and implementation details (e.g., the machine learning method or factor model used).

- Backtesting results:
    
    - Charts: Visualizations such as cumulative performance, rolling 3-year returns, etc.
    - Descriptive statistics: Key statistics such as mean, standard deviation, drawdown, turnover, and Sharpe ratio (or any other relevant metric) for the full backtest period as well as for subperiods (e.g., the last 5 years, or during bull vs. bear market phases).
    - Compare your strategy against the SPI.


## Research Design

This notebook implements a disciplined Swiss equity strategy in four layers:

1. Define an investable universe with liquidity, data quality, and signal-availability screens.
2. Engineer economically motivated cross-sectional signals from prices and JKP fundamentals.
3. Train an expanding-window learning-to-rank model to produce out-of-sample stock scores.
4. Convert those scores into Black-Litterman views and optimize a constrained long-only portfolio.

The implementation is modular by design: every cell owns one research step, paths are resolved from the repository root, and the final cell runs the full pipeline end to end.


In [ ]:
import sys
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, Optional

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor

try:
    import xgboost as xgb
except ImportError:
    xgb = None

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.options.display.float_format = "{:,.4f}".format


## Project Paths

The notebook should run from the repository root, from `assignments/`, or from any child directory of the repository. We therefore infer the project root instead of hard-coding a user-specific absolute path.


In [ ]:
def find_project_root(start: Optional[Path] = None) -> Path:
    """Return the repository root containing the course `src` and `data` folders."""
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not find the qpmwp-course root containing src/ and data/.")


PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"

for path in (PROJECT_ROOT, SRC_DIR):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

print(f"Project root: {PROJECT_ROOT}")
print(f"Data folder:  {DATA_DIR}")
print(f"Results:      {RESULTS_DIR}")


## Course Framework

We reuse the assignment framework for data alignment, backtesting, selection builders, optimization inputs, constraints, and the Black-Litterman optimizer.


In [ ]:
from helper_functions import align_market_data_with_jkp_data, load_data_spi
from estimation.covariance import Covariance
from optimization.optimization import BlackLitterman
from backtesting.backtest import Backtest
from backtesting.backtest_data import BacktestData
from backtesting.backtest_service import BacktestService
from backtesting.backtest_item_builder.bib_classes import (
    OptimizationItemBuilder,
    SelectionItemBuilder,
)
from backtesting.backtest_item_builder.bibfn_constraints import (
    bibfn_box_constraints,
    bibfn_budget_constraint,
    bibfn_size_dependent_upper_bounds,
    bibfn_turnover_constraint,
)
from backtesting.backtest_item_builder.bibfn_optimization_data import (
    bibfn_cap_weights,
    bibfn_return_series,
    bibfn_scores,
)
from backtesting.backtest_item_builder.bibfn_selection import (
    bibfn_selection_gaps,
    bibfn_selection_jkp_data_scores,
    bibfn_selection_min_volume,
)


## Configuration

All research assumptions live in one configuration object. This makes sensitivity checks easy: change a date range, turnover limit, signal set, or model parameter and rerun the pipeline.


In [ ]:
@dataclass(frozen=True)
class StrategyConfig:
    data_dir: Path = DATA_DIR
    results_dir: Path = RESULTS_DIR

    start_date: str = "2003-01-31"
    end_date: Optional[str] = "2022-12-31"
    rebalance_months: int = 3
    lookback_days: int = 365 * 3

    min_volume: float = 500_000
    volume_window_days: int = 365
    max_zero_volume_gap_days: int = 10

    forward_return_months: int = 3
    min_train_dates: int = 24

    fixed_costs_annual: float = 0.01
    transaction_costs_per_rebalancing: float = 0.002

    stock_upper_bound: float = 0.10
    turnover_limit: float = 0.15

    tau_psi: Optional[float] = 0.02
    tau_omega: Optional[float] = 0.05
    view_gen_algo: str = "quintile_sort"
    solver_name: str = "cvxopt"

    jkp_features: tuple[str, ...] = (
        "qmj",
        "value",
        "investment",
    )
    technical_features: tuple[str, ...] = (
        "momentum",
        "mean_reversion_1m",
        "low_volatility",
        "risk_adjusted_momentum",
    )

    xgb_n_estimators: int = 150
    xgb_max_depth: int = 3
    xgb_learning_rate: float = 0.03
    xgb_subsample: float = 0.80
    xgb_colsample_bytree: float = 0.80
    random_state: int = 42


CONFIG = StrategyConfig()
CONFIG


## Cross-Sectional Utilities

These helpers control outliers and scaling. For a stock-selection model, the level of a feature is less important than where a stock sits relative to its peers on the same date.


In [ ]:
def ensure_directory(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)


def winsorize_cross_section(series: pd.Series, lower: float = 0.01, upper: float = 0.99) -> pd.Series:
    """Clip a date-wise cross-section to reduce the influence of extreme observations."""
    clean = series.dropna()
    if clean.empty:
        return series
    lo, hi = clean.quantile([lower, upper])
    return series.clip(lower=lo, upper=hi)


def zscore_cross_section(series: pd.Series) -> pd.Series:
    """Standardize a date-wise cross-section."""
    std = series.std(skipna=True)
    if pd.isna(std) or std == 0:
        return series * 0
    return (series - series.mean(skipna=True)) / std


def normalize_panel(panel: pd.DataFrame, columns: Iterable[str]) -> pd.DataFrame:
    """Winsorize and z-score selected columns date by date."""
    out = panel.copy()
    for column in columns:
        out[column] = (
            out[column]
            .groupby(level="date", group_keys=False)
            .apply(winsorize_cross_section)
            .groupby(level="date", group_keys=False)
            .apply(zscore_cross_section)
        )
    return out


## Data Loading and Sanity Checks

We load the three assignment datasets and standardize date levels. The market data may contain duplicate `(date, id)` observations, so downstream helpers defensively keep the last observation.


In [ ]:
def set_datetime_level(frame: pd.DataFrame, level: str = "date") -> pd.DataFrame:
    """Return a copy whose date index level is a DatetimeIndex."""
    if not isinstance(frame.index, pd.MultiIndex) or level not in frame.index.names:
        return frame
    out = frame.copy()
    level_number = out.index.names.index(level)
    out.index = out.index.set_levels(pd.to_datetime(out.index.levels[level_number]), level=level)
    return out.sort_index()


def load_research_data(config: StrategyConfig) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series]:
    market_data = pd.read_parquet(config.data_dir / "market_data.parquet")
    jkp_data = pd.read_parquet(config.data_dir / "jkp_data.parquet")
    spi = load_data_spi(path=config.data_dir)

    market_data = set_datetime_level(market_data)
    jkp_data = set_datetime_level(jkp_data)
    spi.index = pd.DatetimeIndex(spi.index)

    return market_data, jkp_data, spi.sort_index()


market_data_raw, jkp_data_raw, spi_raw = load_research_data(CONFIG)

data_summary = pd.DataFrame(
    {
        "rows": [len(market_data_raw), len(jkp_data_raw), len(spi_raw)],
        "start": [
            market_data_raw.index.get_level_values("date").min(),
            jkp_data_raw.index.get_level_values("date").min(),
            spi_raw.index.min(),
        ],
        "end": [
            market_data_raw.index.get_level_values("date").max(),
            jkp_data_raw.index.get_level_values("date").max(),
            spi_raw.index.max(),
        ],
        "n_assets": [
            market_data_raw.index.get_level_values("id").nunique(),
            jkp_data_raw.index.get_level_values("id").nunique(),
            np.nan,
        ],
    },
    index=["market_data", "jkp_data", "spi"],
)

data_summary


## Price and Return Matrices

The backtest consumes long-form market data, while feature engineering is easier in wide matrices. This helper creates both prices and daily returns robustly.


In [ ]:
def get_price_and_return_matrices(market_data: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Infer wide price and daily-return matrices from long-form market data."""
    md = market_data.copy().sort_index()
    if md.index.duplicated().any():
        duplicate_count = int(md.index.duplicated().sum())
        print(f"Dropping {duplicate_count:,} duplicate market-data rows.")
        md = md[~md.index.duplicated(keep="last")]

    price_col = next((col for col in ("price", "prc", "close", "adj_close") if col in md.columns), None)
    return_col = next((col for col in ("ret", "return", "returns") if col in md.columns), None)

    prices = md[price_col].unstack("id").sort_index() if price_col else None
    returns = md[return_col].unstack("id").sort_index() if return_col else None

    if prices is not None:
        prices = prices.loc[:, pd.Index(prices.columns).notna()]
        if prices.columns.duplicated().any():
            prices = prices.loc[:, ~prices.columns.duplicated(keep="last")]
    if returns is not None:
        returns = returns.loc[:, pd.Index(returns.columns).notna()]
        if returns.columns.duplicated().any():
            returns = returns.loc[:, ~returns.columns.duplicated(keep="last")]

    if returns is None and prices is not None:
        returns = prices.pct_change()
    if prices is None and returns is not None:
        prices = (1 + returns.fillna(0)).cumprod()

    if prices is None or returns is None:
        raise ValueError(f"Could not infer prices or returns from columns: {list(md.columns)}")

    return prices, returns


## Signal Engineering

The learning-to-rank model receives a compact set of economically interpretable factors. Assignment 4 documents the available JKP factor themes, so we use the broad JKP composites where possible and complement them with price-based signals computed from the market data.

- Quality: `qmj` rewards firms with stronger profitability, growth, and safety characteristics.
- Value: `value` rewards cheaper firms relative to accounting fundamentals. If the broad JKP composite is not supplied directly, the notebook builds it from available Assignment 4 value fields such as book-to-market, assets-to-market, cash-flow yield, and sales-to-market.
- Investment: `investment` captures the empirical investment factor, where more conservative investment behavior is often associated with stronger future returns. If the broad JKP composite is not supplied directly, the notebook builds it from available investment-growth, capital-expenditure-growth, and issuance fields, with lower investment intensity mapped to a higher score.
- Momentum / trend: `momentum` averages 6-month momentum and 12-month momentum excluding the most recent month, following the standard idea that medium-term winners can continue to outperform.
- Defensive / risk: `low_volatility` and `risk_adjusted_momentum` prefer stocks whose trend is not simply compensation for recent volatility.
- Mean reversion: `mean_reversion_1m` is the simple innovative factor in this version. It is the negative of the past 21-trading-day return, so recent short-term losers receive a higher score. The signal is inspired by short-term reversal evidence, but intentionally kept simple and auditable before adding residual or liquidity-conditioned variants.


In [ ]:
def align_wide_factor_to_jkp_dates(factor: pd.DataFrame, jkp_dates: pd.DatetimeIndex) -> pd.Series:
    """Forward-fill a wide daily factor to monthly JKP dates and return a stacked panel."""
    aligned = factor.reindex(factor.index.union(jkp_dates)).sort_index().ffill().loc[jkp_dates]
    return aligned.stack()


def build_technical_features(market_data: pd.DataFrame, jkp_dates: pd.DatetimeIndex) -> pd.DataFrame:
    """Create normalized technical features on the same date-id grid as JKP data."""
    prices, daily_returns = get_price_and_return_matrices(market_data)

    momentum_6m = prices.pct_change(126)
    momentum_12m = prices.pct_change(252)
    momentum_1m = prices.pct_change(21)
    momentum_12m_ex_1m = (1 + momentum_12m) / (1 + momentum_1m) - 1
    momentum = 0.5 * momentum_6m + 0.5 * momentum_12m_ex_1m

    mean_reversion_1m = -prices.pct_change(21)
    volatility_3m = daily_returns.rolling(63).std()
    low_volatility = -volatility_3m
    risk_adjusted_momentum = momentum / (volatility_3m + 1e-6)

    raw_features = {
        "momentum": momentum,
        "mean_reversion_1m": mean_reversion_1m,
        "low_volatility": low_volatility,
        "risk_adjusted_momentum": risk_adjusted_momentum,
    }

    feature_panel = pd.concat(
        {name: align_wide_factor_to_jkp_dates(values, jkp_dates) for name, values in raw_features.items()},
        axis=1,
    )
    feature_panel.index.names = ["date", "id"]
    return normalize_panel(feature_panel, feature_panel.columns)


## Learning Dataset

The target is the future 3-month total return. For each JKP date, we align the latest available market return target and then convert future returns into date-wise decile labels for learning-to-rank.


In [ ]:
JKP_VALUE_COMPONENTS = (
    "at_me",
    "be_me",
    "bev_mev",
    "debt_me",
    "div12m_me",
    "ebitda_mev",
    "fcf_me",
    "ni_me",
    "ocf_me",
    "sale_me",
)

JKP_INVESTMENT_COMPONENTS = (
    "inv_gr1",
    "inv_gr1a",
    "ppeinv_gr1a",
    "capx_gr1",
    "capx_gr2",
    "capx_gr3",
    "eqnetis_at",
    "netis_at",
)


def add_jkp_composite_features(jkp_data: pd.DataFrame) -> pd.DataFrame:
    """Create broad value and investment composites when JKP does not provide them."""
    out = jkp_data.copy()

    if "value" not in out.columns:
        value_components = [col for col in JKP_VALUE_COMPONENTS if col in out.columns]
        if value_components:
            value_panel = normalize_panel(out[value_components], value_components)
            out["value"] = value_panel.mean(axis=1, skipna=True)

    if "investment" not in out.columns:
        investment_components = [col for col in JKP_INVESTMENT_COMPONENTS if col in out.columns]
        if investment_components:
            investment_panel = normalize_panel(out[investment_components], investment_components)
            out["investment"] = -investment_panel.mean(axis=1, skipna=True)

    return out


def compute_forward_returns(market_data: pd.DataFrame, horizon_days: int) -> pd.DataFrame:
    """Compute forward compounded returns over a fixed trading-day horizon."""
    _, daily_returns = get_price_and_return_matrices(market_data)
    gross_forward = (1 + daily_returns).rolling(
        horizon_days,
        min_periods=max(10, horizon_days // 2),
    ).apply(np.prod, raw=True)
    return gross_forward.shift(-horizon_days) - 1


def align_target_to_jkp_dates(forward_returns: pd.DataFrame, jkp_dates: pd.DatetimeIndex) -> pd.DataFrame:
    """Sample the latest available forward-return row at each JKP date."""
    market_dates = forward_returns.index.sort_values()
    rows = []
    for date in jkp_dates:
        eligible_dates = market_dates[market_dates <= date]
        if len(eligible_dates) == 0:
            continue
        row = forward_returns.loc[eligible_dates[-1]].copy()
        row.name = pd.to_datetime(date)
        rows.append(row)

    if not rows:
        raise ValueError("No forward-return targets could be aligned to JKP dates.")

    target = pd.DataFrame(rows)
    target.index.name = "date"
    target = target.stack().rename("forward_return_3m").to_frame()
    target.index.names = ["date", "id"]
    return target


def build_learning_dataset(
    market_data: pd.DataFrame,
    jkp_data: pd.DataFrame,
    config: StrategyConfig,
) -> tuple[pd.DataFrame, list[str]]:
    """Merge fundamental and technical features, then attach future-return labels."""
    jkp_data = add_jkp_composite_features(jkp_data.sort_index())
    jkp_dates = jkp_data.index.get_level_values("date").unique().sort_values()

    available_jkp_features = [col for col in config.jkp_features if col in jkp_data.columns]
    missing_features = sorted(set(config.jkp_features) - set(available_jkp_features))
    if missing_features:
        print(f"Skipping unavailable JKP features: {missing_features}")

    technical_features = build_technical_features(market_data, jkp_dates)
    feature_panel = jkp_data[available_jkp_features].join(technical_features, how="left")

    feature_columns = available_jkp_features + [
        col for col in config.technical_features if col in feature_panel.columns
    ]
    feature_panel = normalize_panel(feature_panel, feature_columns)

    horizon_days = int(config.forward_return_months * 21)
    forward_returns = compute_forward_returns(market_data, horizon_days=horizon_days)
    target = align_target_to_jkp_dates(forward_returns, jkp_dates)

    dataset = feature_panel.join(target, how="left")
    dataset = dataset.replace([np.inf, -np.inf], np.nan)
    return dataset, feature_columns


learning_dataset, feature_columns = build_learning_dataset(market_data_raw, jkp_data_raw, CONFIG)

print(f"Feature columns ({len(feature_columns)}): {feature_columns}")
learning_dataset[feature_columns + ["forward_return_3m"]].head()


## Expanding-Window Learning to Rank

At each prediction date, the model trains only on prior dates. This is the core anti-lookahead rule. The preferred estimator is `XGBRanker`; if `xgboost` is unavailable, the notebook falls back to a scikit-learn gradient-boosted tree trained on the same rank labels. The model learns relative stock attractiveness, not a fragile point forecast of exact returns.


In [ ]:
def add_ranking_labels(dataset: pd.DataFrame, target_col: str = "forward_return_3m") -> pd.DataFrame:
    """Convert future returns into non-negative date-wise relevance labels."""
    out = dataset.copy()

    def rank_to_decile(cross_section: pd.Series) -> pd.Series:
        clean = cross_section.dropna()
        if clean.nunique() < 2:
            return pd.Series(np.nan, index=cross_section.index)
        ranks = cross_section.rank(method="first", ascending=True)
        labels = pd.qcut(ranks, q=10, labels=False, duplicates="drop")
        return labels.astype(float)

    out["rank_label"] = out.groupby(level="date")[target_col].transform(rank_to_decile)
    return out


def make_rank_model(config: StrategyConfig) -> tuple[object, bool]:
    """Return a tree model and whether it expects XGBoost ranking groups."""
    if xgb is None:
        model = HistGradientBoostingRegressor(
            max_iter=config.xgb_n_estimators,
            max_leaf_nodes=2 ** config.xgb_max_depth,
            learning_rate=config.xgb_learning_rate,
            l2_regularization=1.0,
            random_state=config.random_state,
        )
        return model, False

    model = xgb.XGBRanker(
        objective="rank:pairwise",
        eval_metric="ndcg",
        n_estimators=config.xgb_n_estimators,
        max_depth=config.xgb_max_depth,
        learning_rate=config.xgb_learning_rate,
        subsample=config.xgb_subsample,
        colsample_bytree=config.xgb_colsample_bytree,
        reg_alpha=0.10,
        reg_lambda=1.00,
        random_state=config.random_state,
        n_jobs=-1,
        tree_method="hist",
    )
    return model, True


def generate_out_of_sample_signal(
    dataset: pd.DataFrame,
    feature_columns: list[str],
    config: StrategyConfig,
) -> pd.Series:
    """Train an expanding-window ranker and return an out-of-sample score panel."""
    ranked_dataset = add_ranking_labels(dataset)
    dates = ranked_dataset.index.get_level_values("date").unique().sort_values()
    prediction_dates = dates[dates >= pd.Timestamp(config.start_date)]
    if config.end_date is not None:
        prediction_dates = prediction_dates[prediction_dates <= pd.Timestamp(config.end_date)]
    predictions: list[pd.Series] = []

    for prediction_index, prediction_date in enumerate(prediction_dates, start=1):
        date_index = dates.get_loc(prediction_date)
        train_dates = dates[:date_index]
        if len(train_dates) < config.min_train_dates:
            continue

        train = ranked_dataset.loc[ranked_dataset.index.get_level_values("date").isin(train_dates)]
        test = ranked_dataset.loc[ranked_dataset.index.get_level_values("date") == prediction_date]

        train = train.dropna(subset=feature_columns + ["rank_label"])
        test = test.dropna(subset=feature_columns)
        if train.empty or test.empty:
            continue

        group_sizes = train.groupby(train.index.get_level_values("date")).size().to_numpy()
        model, uses_ranking_groups = make_rank_model(config)
        if uses_ranking_groups:
            model.fit(
                train[feature_columns].astype(float),
                train["rank_label"].astype(float),
                group=group_sizes,
                verbose=False,
            )
        else:
            model.fit(
                train[feature_columns].astype(float),
                train["rank_label"].astype(float),
            )

        prediction = pd.Series(
            model.predict(test[feature_columns].astype(float)),
            index=test.index,
            name="ml_signal",
        )
        predictions.append(prediction)

        if len(predictions) == 1 or len(predictions) % 12 == 0:
            date_label = pd.to_datetime(prediction_date).date()
            print(f"Generated signal for {date_label} ({prediction_index}/{len(prediction_dates)})")

    if not predictions:
        raise RuntimeError("No ML predictions were generated. Check data availability and min_train_dates.")

    signal = pd.concat(predictions).sort_index()
    signal = (
        signal
        .groupby(level="date", group_keys=False)
        .apply(winsorize_cross_section)
        .groupby(level="date", group_keys=False)
        .apply(zscore_cross_section)
    )
    signal.name = "ml_signal"
    return signal


## Portfolio Construction

The ML score is not used as a portfolio weight directly. It becomes a Black-Litterman view, which is then balanced against covariance risk, market-cap priors, liquidity screens, concentration limits, and turnover control.


In [ ]:
def build_rebalance_dates(
    market_data: pd.DataFrame,
    jkp_data: pd.DataFrame,
    config: StrategyConfig,
) -> list[str]:
    market_dates = market_data.index.get_level_values("date").unique().sort_values()
    jkp_dates = jkp_data.index.get_level_values("date").unique().sort_values()

    dates = jkp_dates[jkp_dates > market_dates.min()][:: config.rebalance_months]
    rebalance_dates = dates.strftime("%Y-%m-%d").tolist()
    rebalance_dates = [date for date in rebalance_dates if date >= config.start_date]
    if config.end_date is not None:
        rebalance_dates = [date for date in rebalance_dates if date <= config.end_date]

    if len(rebalance_dates) > 1:
        rebalance_dates = rebalance_dates[:-1]
    if not rebalance_dates:
        raise ValueError("No rebalance dates available after applying the configured date range.")
    return rebalance_dates


def build_backtest_service(
    market_data: pd.DataFrame,
    jkp_data_with_signal: pd.DataFrame,
    spi: pd.Series,
    rebalance_dates: list[str],
    config: StrategyConfig,
) -> BacktestService:
    data = BacktestData()
    data.market_data = market_data
    data.jkp_data = jkp_data_with_signal
    data.bm_series = spi

    selection_builders = {
        "gaps": SelectionItemBuilder(
            bibfn=bibfn_selection_gaps,
            width=config.lookback_days,
            n_days=config.max_zero_volume_gap_days,
        ),
        "min_volume": SelectionItemBuilder(
            bibfn=bibfn_selection_min_volume,
            width=config.volume_window_days,
            min_volume=config.min_volume,
            agg_fn=np.median,
        ),
        "jkp_data_scores": SelectionItemBuilder(
            bibfn=bibfn_selection_jkp_data_scores,
            fields=["ml_signal"],
        ),
    }

    optimization_builders = {
        "return_series": OptimizationItemBuilder(
            bibfn=bibfn_return_series,
            width=config.lookback_days,
            weekdays_only=True,
            fillna_value=0,
        ),
        "cap_weights": OptimizationItemBuilder(bibfn=bibfn_cap_weights),
        "scores": OptimizationItemBuilder(bibfn=bibfn_scores, fields=["ml_signal"]),
        "budget_constraint": OptimizationItemBuilder(bibfn=bibfn_budget_constraint, budget=1),
        "box_constraints": OptimizationItemBuilder(
            bibfn=bibfn_box_constraints,
            lower=0,
            upper=config.stock_upper_bound,
        ),
        "size_dependent_upper_bounds": OptimizationItemBuilder(
            bibfn=bibfn_size_dependent_upper_bounds,
            small_cap={"threshold": 300_000_000, "upper": 0.02},
            mid_cap={"threshold": 1_000_000_000, "upper": 0.05},
            large_cap={"threshold": 10_000_000_000, "upper": config.stock_upper_bound},
        ),
        "turnover_constraint": OptimizationItemBuilder(
            bibfn=bibfn_turnover_constraint,
            turnover_limit=config.turnover_limit,
        ),
    }

    tau_default = 1 / config.lookback_days
    optimization = BlackLitterman(
        covariance=Covariance(method="pearson", check_positive_definite=True),
        tau_psi=config.tau_psi or tau_default,
        tau_omega=config.tau_omega or tau_default,
        view_gen_algo=config.view_gen_algo,
        use_unconditional_cov=True,
        signal_names=["ml_signal"],
        solver_name=config.solver_name,
    )

    return BacktestService(
        data=data,
        optimization=optimization,
        selection_item_builders=selection_builders,
        optimization_item_builders=optimization_builders,
        rebdates=rebalance_dates,
        quiet=False,
    )


## Performance Analytics

The reporting layer separates gross strategy returns, net strategy returns, benchmark returns, turnover, and standard performance statistics.


In [ ]:
def relative_simple_returns(returns: pd.DataFrame, benchmark: pd.Series) -> pd.DataFrame:
    """Simple active return adjusted for the benchmark base return."""
    return returns.subtract(benchmark, axis=0).divide(1 + benchmark, axis=0)


def apply_costs_to_returns(
    gross_returns: pd.Series,
    turnover: pd.Series,
    fixed_costs_annual: float,
    transaction_costs_per_rebalancing: float,
    n_days_per_year: int = 252,
) -> pd.Series:
    """Apply fixed and transaction costs without relying on pandas positional indexing in Strategy.simulate."""
    net_returns = gross_returns.copy()

    if transaction_costs_per_rebalancing != 0:
        variable_costs = turnover * transaction_costs_per_rebalancing
        common_dates = variable_costs.index.intersection(net_returns.index)
        net_returns.loc[common_dates] -= variable_costs.loc[common_dates]

    if fixed_costs_annual != 0 and len(net_returns) > 1:
        elapsed_days = (net_returns.index[1:] - net_returns.index[:-1]).days
        fixed_costs = (1 + fixed_costs_annual) ** (elapsed_days / n_days_per_year) - 1
        net_returns.iloc[1:] -= fixed_costs

    return net_returns


def performance_table(returns: pd.DataFrame, benchmark_col: str = "Benchmark") -> pd.DataFrame:
    """Compute standard annualized performance and benchmark-relative metrics."""
    periods_per_year = 252
    metrics = {}

    for column in returns.columns:
        asset_returns = returns[column].dropna()
        common = returns[[column, benchmark_col]].dropna()

        cumulative_return = (1 + asset_returns).prod() - 1
        n_periods = len(asset_returns)
        annual_return = (1 + cumulative_return) ** (periods_per_year / n_periods) - 1
        annual_volatility = asset_returns.std() * np.sqrt(periods_per_year)
        sharpe_ratio = annual_return / annual_volatility if annual_volatility != 0 else np.nan

        wealth = (1 + asset_returns).cumprod()
        drawdown = wealth / wealth.cummax() - 1

        if column == benchmark_col:
            tracking_error = 0.0
            alpha = 0.0
            beta = 1.0
        else:
            active_return = common[column] - common[benchmark_col]
            tracking_error = active_return.std() * np.sqrt(periods_per_year)
            x = common[benchmark_col].to_numpy()
            y = common[column].to_numpy()
            beta = np.cov(y, x)[0, 1] / np.var(x) if np.var(x) != 0 else np.nan
            alpha = (y.mean() - beta * x.mean()) * periods_per_year

        metrics[column] = {
            "Annual Return": annual_return,
            "Cumulative Return": cumulative_return,
            "Annual Volatility": annual_volatility,
            "Sharpe Ratio": sharpe_ratio,
            "Max Drawdown": drawdown.min(),
            "Tracking Error": tracking_error,
            "Alpha": alpha,
            "Beta": beta,
        }

    return pd.DataFrame(metrics).T


def save_performance_plots(returns: pd.DataFrame, turnover: pd.Series, results_dir: Path) -> None:
    """Persist cumulative performance, active performance, and turnover plots."""
    ensure_directory(results_dir)

    ax = np.log1p(returns).cumsum().plot(figsize=(11, 6), title="Cumulative Log Performance")
    ax.set_ylabel("Cumulative log return")
    plt.tight_layout()
    plt.savefig(results_dir / "cumulative_log_performance.png", dpi=150)
    plt.close()

    active_returns = relative_simple_returns(returns, returns["Benchmark"])
    ax = np.log1p(active_returns).cumsum().plot(figsize=(11, 6), title="Cumulative Out-/Underperformance vs SPI")
    ax.set_ylabel("Cumulative relative log return")
    plt.tight_layout()
    plt.savefig(results_dir / "relative_performance_vs_spi.png", dpi=150)
    plt.close()

    ax = turnover.plot(figsize=(11, 5), title="Portfolio Turnover")
    ax.set_ylabel("Turnover")
    plt.tight_layout()
    plt.savefig(results_dir / "turnover.png", dpi=150)
    plt.close()


## End-to-End Research Pipeline

This function wires the notebook together. It returns the main research artifacts so that diagnostics can be added without rerunning every intermediate step manually.


In [ ]:
def run_strategy_research(config: StrategyConfig = CONFIG) -> dict[str, object]:
    ensure_directory(config.results_dir)

    print("1/8 Loading data")
    market_data, jkp_data, spi = load_research_data(config)

    print("2/8 Building ML dataset")
    dataset, features = build_learning_dataset(market_data, jkp_data, config)
    print(f"Using {len(features)} features: {features}")

    print("3/8 Training expanding-window ranker")
    ml_signal = generate_out_of_sample_signal(dataset, features, config)
    ml_signal.to_frame().to_parquet(config.results_dir / "ltr_signal.parquet")

    print("4/8 Aligning market data with JKP signal panel")
    jkp_with_signal = jkp_data.join(ml_signal.to_frame(), how="left")
    market_data_aligned, jkp_with_signal = align_market_data_with_jkp_data(
        market_data=market_data,
        jkp_data=jkp_with_signal,
    )

    print("5/8 Building rebalance calendar")
    rebalance_dates = build_rebalance_dates(market_data_aligned, jkp_with_signal, config)
    print(f"Rebalance dates: {len(rebalance_dates)} from {rebalance_dates[0]} to {rebalance_dates[-1]}")

    print("6/8 Running constrained Black-Litterman backtest")
    backtest_service = build_backtest_service(
        market_data=market_data_aligned,
        jkp_data_with_signal=jkp_with_signal,
        spi=spi,
        rebalance_dates=rebalance_dates,
        config=config,
    )
    backtest = Backtest()
    backtest.run(bs=backtest_service)
    backtest.save(path=config.results_dir, filename="bt_ml_black_litterman.pickle")

    print("7/8 Simulating gross and net strategy returns")
    return_series = backtest_service.data.get_return_series(weekdays_only=False)
    gross_returns = backtest.strategy.simulate(return_series=return_series, fc=0, vc=0)
    turnover_for_costs = backtest.strategy.turnover(return_series=return_series, rescale=False)
    net_returns = apply_costs_to_returns(
        gross_returns=gross_returns,
        turnover=turnover_for_costs,
        fixed_costs_annual=config.fixed_costs_annual,
        transaction_costs_per_rebalancing=config.transaction_costs_per_rebalancing,
    )

    strategy_returns = pd.concat(
        {
            "Benchmark": backtest_service.data.bm_series,
            "ML-BL Gross": gross_returns,
            "ML-BL Net": net_returns,
        },
        axis=1,
    ).dropna()
    if config.end_date is not None:
        strategy_returns = strategy_returns[strategy_returns.index <= config.end_date]
    strategy_returns.to_csv(config.results_dir / "simulation_returns.csv")

    print("8/8 Computing diagnostics and saving outputs")
    turnover = backtest.strategy.turnover(return_series=return_series)
    turnover.to_csv(config.results_dir / "turnover.csv", header=["turnover"])

    metrics = performance_table(strategy_returns, benchmark_col="Benchmark")
    metrics.to_csv(config.results_dir / "performance_metrics.csv")
    save_performance_plots(strategy_returns, turnover, config.results_dir)

    print(f"Done. Outputs saved in: {config.results_dir.resolve()}")
    return {
        "config": config,
        "features": features,
        "learning_dataset": dataset,
        "ml_signal": ml_signal,
        "rebalance_dates": rebalance_dates,
        "backtest_service": backtest_service,
        "backtest": backtest,
        "returns": strategy_returns,
        "turnover": turnover,
        "metrics": metrics,
    }


## Run the Strategy

Execute the cell below to run the full research pipeline. The final object, `research`, contains the backtest, return series, turnover, metrics, and intermediate signal data.


In [ ]:
research = run_strategy_research(CONFIG)
research["metrics"].round(4)


## Strategy Intuition

The strategy is designed to avoid turning a noisy return forecast into a brittle portfolio. The machine-learning layer only ranks stocks cross-sectionally, which is usually more stable than forecasting exact returns. Black-Litterman then translates those rankings into relative views while still anchoring the portfolio to market-cap priors and recent covariance risk.

Economically, the model combines five complementary effects: JKP quality, value, and investment characteristics; medium-term trend; defensive risk control; and a short-term mean-reversion signal based on the past 21 trading days. The mean-reversion feature is intentionally simple in this version: it asks whether a recent one-month loser is likely to recover relative to peers, while the ranker learns whether that signal is useful alongside fundamentals, trend, and risk features.

The optimizer then asks a separate question: given those ranked views, what portfolio is feasible after liquidity, concentration, size, and turnover constraints? That separation between alpha research and portfolio construction is what makes the workflow more professional and easier to audit.

### Future Mean-Reversion Improvements

The current `mean_reversion_1m` factor is a deliberately transparent first version. Natural extensions include:

- Residual reversal after removing market and sector returns, so the factor captures idiosyncratic overreaction rather than broad market moves.
- Liquidity-conditioned reversal, where recent loser scores are stronger when the move coincides with temporary price pressure or abnormal trading activity.
- Volatility-scaled reversal, so a 5% move in a calm stock is treated differently from a 5% move in a very volatile stock.
- Sector-neutral reversal, ranking stocks against sector peers before combining them in the global cross-section.
- Extreme-move reversal, focusing only on unusually large short-term dislocations instead of all recent returns.
- Regime-conditioned reversal, reducing reversal exposure when the market is trending strongly and short-term continuation is more likely.
- Horizon variants such as 5d, 10d, and 21d reversal features, letting the model learn which mean-reversion horizon matters most.
